## 1. Configuración e Importaciones
En esta celda importamos las librerías y definimos las constantes del proyecto (nombres de datasets, algoritmos, etc.).

In [9]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import os
from scipy import stats
from sklearn.preprocessing import LabelEncoder, label_binarize
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix, roc_auc_score, roc_curve, auc
from itertools import cycle
import pathlib as pl

# CONFIGURACIÓN DE RUTAS ---
# AJUSTAR estas rutas según estructura de nuestras carpetas
TYPES = ['original', 'estandarizado', 'normalizado']
VARIANTS = ['', '_PCA95', '_PCA80']
FOLDER_PREFIX = 'conj'
VALIDATING_FILE_REGEX = 'validating*.csv'
DATA_PATH = './kfolds_data'
MODEL_PATH = './trained_models'
MODEL_EXT = 'joblib'                       
PATH_PREDICCIONES = "./predicciones"                                   # Ruta donde se guardarán las predicciones  
PATH_METRICAS = "./metricas"                                           # Ruta donde se guardarán las métricas                

# Crear carpetas si no existen
os.makedirs(PATH_PREDICCIONES, exist_ok=True)
os.makedirs(PATH_METRICAS, exist_ok=True)

# Lista con los modelos a evaluar y los tipos de ensemble
MODELOS = ["KNN", "SVM", "NaiveBayes", "RandomForest"]
ENSEMBLES = ["Ensemble_Votacion", "Ensemble_Media", "Ensemble_Mediana"]

## 2. Funciones Auxiliares (Carga y Guardado)
Necesitamos funciones para cargar los datos y modelos correspondientes a cada iteración y almacenar los resultados.

In [10]:
# Funciones de carga, evaluación y guardado de resultados
def cargar_datos_test(filename):
    try:
        df = pd.read_csv(filename)
        X = df.drop('species', axis=1)
        Y = df['species']
        return X, Y
    except FileNotFoundError:
        print(f"Falta archivo {filename}")
        return None, None

def cargar_modelo(filename):
    try:
        return joblib.load(filename)
    except FileNotFoundError:
        print(f"Falta modelo {filename}")
        return None

def plot_multiclass_roc(y_true, y_proba, dataset_name, method_name, save_path):
    clases = ['setosa', 'versicolor', 'virginica']
    n_classes = len(clases)
    
    y_true_bin = label_binarize(y_true, classes=clases)

    fpr = dict()
    tpr = dict()
    roc_auc = dict()
    
    for i in range(n_classes):
        fpr[i], tpr[i], _ = roc_curve(y_true_bin[:, i], y_proba[:, i])
        roc_auc[i] = auc(fpr[i], tpr[i])

    # Configurar el gráfico
    plt.figure(figsize=(8, 6))
    colores = cycle(['blue', 'red', 'green'])
    
    for i, color in zip(range(n_classes), colores):
        plt.plot(fpr[i], tpr[i], color=color, lw=2,
                 label=f'ROC de {clases[i]} (AUC = {roc_auc[i]:.2f})')

    plt.plot([0, 1], [0, 1], 'k--', lw=2) # Línea diagonal (azar)
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('Tasa de Falsos Positivos (FPR)')
    plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
    plt.title(f'Curva ROC Multiclase - {method_name} ({dataset_name})')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3)
    
    plt.savefig(save_path, bbox_inches='tight')
    plt.close()


def calcular_metricas(y_true, y_pred, y_proba):
    """
    Calcula las métricas solicitadas.
    Adapta fórmulas binarias a multiclase usando macro-average.
    """
    # Métricas básicas de sklearn (usando 'macro' para multiclase)
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')      # F1-Score
    rec = recall_score(y_true, y_pred, average='macro') # Sensibilidad
    prec = precision_score(y_true, y_pred, average='macro') # Precisión
    
    # Métricas derivadas de la Matriz de Confusión (Especificidad, FNR, FPR)
    cm = confusion_matrix(y_true, y_pred)
    
    # Cálculo de TP, TN, FP, FN por clase
    FP = cm.sum(axis=0) - np.diag(cm)  
    FN = cm.sum(axis=1) - np.diag(cm)
    TP = np.diag(cm)
    TN = cm.sum() - (FP + FN + TP)

    # Evitar división por cero
    epsilon = 1e-7 
    
    # Promedios macro de las tasas
    spec_macro = np.mean(TN / (FP + TN + epsilon))
    fnr_macro = np.mean(FN / (TP + FN + epsilon))
    fpr_macro = np.mean(FP / (FP + TN + epsilon))
    
    # AUC 
    auc_val = 0
    if y_proba is not None:
        try:
            auc_val = roc_auc_score(y_true, y_proba, multi_class='ovr')
        except:
            pass

    return {
        "Exactitud": acc, "F1": f1, "Sensibilidad": rec, "Recall": rec,
        "Precision": prec, "Especificidad": spec_macro,
        "FNR": fnr_macro, "FPR": fpr_macro, "AUC": auc_val
    }

def guardar_resultados(dataset, fold, metodo, y_true, y_pred, y_proba):
    """
    Guarda DOS archivos: 
     Predicciones crudas (para ensembles y futuros análisis) y métricas calculadas (para la tabla de resultados finales)
    """
    # Guardar Predicciones (CSV Grande)
    data_pred = {"y_true": y_true, "y_pred": y_pred}
    if y_proba is not None:
        for i in range(y_proba.shape[1]):
            data_pred[f"prob_{i}"] = y_proba[:, i]
    
    os.makedirs(f"{PATH_PREDICCIONES}/{metodo}/{dataset}", exist_ok=True)
    pd.DataFrame(data_pred).to_csv(f"{PATH_PREDICCIONES}/{metodo}/{dataset}/{metodo}{fold}_{dataset}_predicts.csv", index=False)

    # Guardar Métricas (CSV Pequeño con una fila)
    metricas = calcular_metricas(y_true, y_pred, y_proba)
    metricas['Dataset'] = dataset
    metricas['Fold'] = fold
    metricas['Method'] = metodo

    # Reordenar para que las columnas identificadoras vayan primero
    cols = ['Dataset', 'Fold', 'Method'] + [k for k in metricas.keys() if k not in ['Dataset', 'Fold', 'Method']]
    
    os.makedirs(f"{PATH_METRICAS}/{metodo}/{dataset}", exist_ok=True)
    pd.DataFrame([metricas], columns=cols).to_csv(f"{PATH_METRICAS}/{metodo}/{dataset}/{metodo}{fold}_{dataset}_metrics.csv", index=False)

    ruta_imagen = f"{PATH_METRICAS}/{metodo}/{dataset}/{metodo}{fold}_{dataset}_ROC.png"
    plot_multiclass_roc(y_true, y_proba, dataset, metodo, ruta_imagen)

## 3. Creación de predicciones y estadisticas de modelos base y ensembles
En esta sección iteramos sobre cada dataset y fold para generar las predicciones y métricas de los modelos base (KNN, SVM, NB, RF) utilizando los datos de test. Además, cuando los cuatro modelos se ejecutan correctamente, calculamos y evaluamos tres métodos de ensemble (Votación, Media y Mediana) combinando sus resultados para intentar mejorar el rendimiento final.

In [11]:
for tipo in TYPES:
    for var in VARIANTS:
        dataset_name = f"{tipo}{var}"
        validating_route = pl.Path(f'{DATA_PATH}/{FOLDER_PREFIX}_{dataset_name}/')
        validating_csv = sorted([x.name for x in validating_route.glob(VALIDATING_FILE_REGEX)])

        for fold, csv_file in enumerate(validating_csv):
            # Cargar datos de test
            X_test, Y_test = cargar_datos_test(validating_route / validating_csv[fold])
            if X_test is None or Y_test is None:
                continue
            
            # Listas para guardar las predicciones de los 4 modelos base para este fold actual
            ensemble_preds_clases = []
            ensemble_preds_probabilidades = []
            modelos_validos_count = 0
            class_labels = None

            for model in MODELOS:
                model_folder = pl.Path(f'{MODEL_PATH}/{model}/{dataset_name}')
                validating_models = sorted([x.name for x in model_folder.glob(f'{model}*.{MODEL_EXT}')])

                # Cargar modelo entrenado
                modelo = cargar_modelo(model_folder / validating_models[fold])
                if modelo is None:
                    continue
                    
                # Guardamos las clases del modelo (necesario para los ensemble)
                if class_labels is None and hasattr(modelo, "classes_"):
                    class_labels = modelo.classes_

                # Realizar predicciones individuales
                Y_pred = modelo.predict(X_test)
                try:
                    Y_proba = modelo.predict_proba(X_test)
                except:
                    Y_proba = None
                    
                # Guardar resultados
                guardar_resultados(dataset_name, fold + 1, model, Y_test, Y_pred, Y_proba)

                # Añadir a las listas para el Ensemble
                ensemble_preds_clases.append(Y_pred)
                ensemble_preds_probabilidades.append(Y_proba)
                modelos_validos_count += 1

            if modelos_validos_count == 4:
                # A) ENSEMBLE VOTACIÓN (Moda de las clases)
                stack_preds = np.stack(ensemble_preds_clases)
                le = LabelEncoder()
                preds_encoded = le.fit_transform(stack_preds.ravel())
                preds_encoded = preds_encoded.reshape(stack_preds.shape)
                moda_resultado = stats.mode(preds_encoded, axis=0, keepdims=True)
                moda_numerica = moda_resultado.mode[0]
                y_pred_votacion = le.inverse_transform(moda_numerica)
                y_proba_votacion = np.mean(np.stack(ensemble_preds_probabilidades), axis=0)
                guardar_resultados(dataset_name, fold + 1, "Ensemble_Votacion", Y_test, y_pred_votacion, y_proba_votacion)

                # B) ENSEMBLE MEDIA (Promedio de probabilidades)
                y_proba_media = np.mean(np.stack(ensemble_preds_probabilidades), axis=0)
                indices_media = np.argmax(y_proba_media, axis=1) # Devuelve 0, 1, 2
                y_pred_media = class_labels[indices_media]       # Traducimos a etiqueta real
                guardar_resultados(dataset_name, fold + 1, "Ensemble_Media", Y_test, y_pred_media, y_proba_media)

                # C) ENSEMBLE MEDIANA (Mediana de probabilidades)
                y_proba_mediana = np.median(np.stack(ensemble_preds_probabilidades), axis=0)
                indices_mediana = np.argmax(y_proba_mediana, axis=1) # Devuelve 0, 1, 2
                y_pred_mediana = class_labels[indices_mediana]       # Traducimos a etiqueta real
                guardar_resultados(dataset_name, fold + 1, "Ensemble_Mediana", Y_test, y_pred_mediana, y_proba_mediana)

            else:
                print(f"\nSolo {modelos_validos_count} modelos validos en fold {fold}. No se ha aplicado Ensemble.")

## 4. Calculo medias de los folds
Este bloque consolida los resultados de la validación cruzada agrupando las métricas de todos los *folds* para cada modelo y ensemble. Posteriormente, calcula y guarda el promedio de dichas métricas en un nuevo archivo, obteniendo así el rendimiento final representativo necesario para el informe.

In [12]:
TODOS_LOS_METODOS = MODELOS + ENSEMBLES

for metodo in TODOS_LOS_METODOS:
    for tipo in TYPES:
        for var in VARIANTS:
            dataset = f"{tipo}{var}"
            metrics_path = pl.Path(f"{PATH_METRICAS}/{metodo}/{dataset}/")
            all_metrics_files = sorted([x.name for x in metrics_path.glob('*.csv')])
            df_all_metrics = pd.concat([pd.read_csv(metrics_path / f) for f in all_metrics_files], ignore_index=True)
            df_mean_metrics = df_all_metrics.drop('Fold', axis=1).mean(numeric_only=True)
            print(f"\nMétricas medias para {metodo} en {dataset}:\n{df_mean_metrics}\n")
            df_mean_metrics.to_frame().T.to_csv(f"{PATH_METRICAS}/{metodo}/{dataset}/{metodo}_{dataset}_average_metrics.csv", index=False)

            lista_preds = []
            for fold in [1, 2, 3, 4, 5]:
                nombre_archivo = f"{metodo}{fold}_{dataset}_predicts.csv"
                ruta_archivo = os.path.join(PATH_PREDICCIONES, metodo, dataset, nombre_archivo)
                lista_preds.append(pd.read_csv(ruta_archivo))
            
            df_global = pd.concat(lista_preds, ignore_index=True)
            y_true_global = df_global['y_true'].values
            y_proba_global = df_global.filter(like='prob_').values
                
            ruta_imagen = f"{PATH_METRICAS}/{metodo}/{dataset}/{metodo}_{dataset}_average_ROC.png"
            plot_multiclass_roc(y_true_global, y_proba_global, dataset, metodo, ruta_imagen)


Métricas medias para KNN en original:
Exactitud        1.0
F1               1.0
Sensibilidad     1.0
Recall           1.0
Precision        1.0
Especificidad    1.0
FNR              0.0
FPR              0.0
AUC              1.0
dtype: float64


Métricas medias para KNN en original_PCA95:
Exactitud        0.966667
F1               0.966515
Sensibilidad     0.966667
Recall           0.966667
Precision        0.969495
Especificidad    0.983333
FNR              0.033333
FPR              0.016667
AUC              0.990667
dtype: float64


Métricas medias para KNN en original_PCA80:
Exactitud        0.920000
F1               0.919765
Sensibilidad     0.920000
Recall           0.920000
Precision        0.923165
Especificidad    0.960000
FNR              0.080000
FPR              0.040000
AUC              0.977667
dtype: float64


Métricas medias para KNN en estandarizado:
Exactitud        0.960000
F1               0.959832
Sensibilidad     0.960000
Recall           0.960000
Precision        0